# Parameter Estimation for REE Distribution Coefficients

This notebook demonstrates using `difflow.estimation` to fit distribution coefficient models to experimental liquid-liquid extraction data for rare earth elements.

## Scenario

We have experimental data from liquid-liquid extraction experiments where we:
1. **Measure** aqueous and organic phase concentrations (via ICP-MS or spectrophotometry)
2. **Fit** the pH-dependent distribution coefficient model to predict these concentrations

The distribution coefficient model is:
$$\log_{10}(D) = a + b \cdot pH + c \cdot pH^2$$

We need to estimate parameters $a$, $b$, and $c$ for each element (La, Nd, Dy).

## Setup

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import pandas as pd

jax.config.update("jax_enable_x64", True)

from difflow.estimation import Estimator, Experiment

## Generate Synthetic Experimental Data

We simulate realistic extraction experiments:
1. Contact aqueous feed with organic solvent at known pH
2. Allow phases to reach equilibrium
3. **Measure concentrations** in both phases (with analytical uncertainty)

This is what you would actually do in the lab!

In [ ]:
# True parameters for each element (unknown to the estimator)
true_params = {
    'La': {'a': -2.5, 'b': 1.8, 'c': -0.15},  # Light REE - moderate extraction
    'Nd': {'a': -1.8, 'b': 2.1, 'c': -0.20},  # Medium REE - better extraction
    'Dy': {'a': -0.9, 'b': 2.5, 'c': -0.25},  # Heavy REE - strong extraction
}

def calculate_D_true(element, pH):
    """Calculate true distribution coefficient."""
    p = true_params[element]
    log_D = p['a'] + p['b'] * pH + p['c'] * pH**2
    return 10**log_D

# Experimental conditions
np.random.seed(42)
pH_values = np.array([2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0])
n_replicates = 3  # Multiple measurements per pH
V_aq = 10.0  # mL aqueous phase
V_org = 10.0  # mL organic phase (equal volumes)
phase_ratio = V_org / V_aq

experimental_data = []

for pH in pH_values:
    for rep in range(n_replicates):
        # Initial aqueous concentrations (mg/L)
        C0_La = 100.0
        C0_Nd = 100.0
        C0_Dy = 100.0
        
        # Calculate true D values at this pH
        D_La_true = calculate_D_true('La', pH)
        D_Nd_true = calculate_D_true('Nd', pH)
        D_Dy_true = calculate_D_true('Dy', pH)
        
        # Mass balance at equilibrium:
        # C0 * V_aq = C_aq * V_aq + C_org * V_org
        # D = C_org / C_aq
        # => C0 = C_aq + C_org * (V_org/V_aq)
        # => C0 = C_aq + D * C_aq * (V_org/V_aq)
        # => C_aq = C0 / (1 + D * V_org/V_aq)
        
        C_La_aq_true = C0_La / (1 + D_La_true * phase_ratio)
        C_Nd_aq_true = C0_Nd / (1 + D_Nd_true * phase_ratio)
        C_Dy_aq_true = C0_Dy / (1 + D_Dy_true * phase_ratio)
        
        C_La_org_true = D_La_true * C_La_aq_true
        C_Nd_org_true = D_Nd_true * C_Nd_aq_true
        C_Dy_org_true = D_Dy_true * C_Dy_aq_true
        
        # Add measurement noise (2% relative error - typical for ICP-MS)
        noise_level = 0.02
        
        # Measured concentrations (what we actually observe in lab)
        C_La_aq_meas = C_La_aq_true * (1 + noise_level * np.random.randn())
        C_Nd_aq_meas = C_Nd_aq_true * (1 + noise_level * np.random.randn())
        C_Dy_aq_meas = C_Dy_aq_true * (1 + noise_level * np.random.randn())
        
        C_La_org_meas = C_La_org_true * (1 + noise_level * np.random.randn())
        C_Nd_org_meas = C_Nd_org_true * (1 + noise_level * np.random.randn())
        C_Dy_org_meas = C_Dy_org_true * (1 + noise_level * np.random.randn())
        
        # Store what we measured
        experimental_data.append({
            'pH': pH,
            'C0_La': C0_La,
            'C0_Nd': C0_Nd,
            'C0_Dy': C0_Dy,
            'C_La_aq': C_La_aq_meas,
            'C_Nd_aq': C_Nd_aq_meas,
            'C_Dy_aq': C_Dy_aq_meas,
            'C_La_org': C_La_org_meas,
            'C_Nd_org': C_Nd_org_meas,
            'C_Dy_org': C_Dy_org_meas,
            'V_aq': V_aq,
            'V_org': V_org,
        })

print(f"Generated {len(experimental_data)} extraction experiments")
print(f"pH range: {pH_values.min():.1f} - {pH_values.max():.1f}")
print(f"Replicates per pH: {n_replicates}")
print(f"\nWhat we measured (example):")
print(f"  pH: {experimental_data[0]['pH']}")
print(f"  C_La_aq: {experimental_data[0]['C_La_aq']:.2f} mg/L")
print(f"  C_La_org: {experimental_data[0]['C_La_org']:.2f} mg/L")
print(f"  Implied D_La: {experimental_data[0]['C_La_org']/experimental_data[0]['C_La_aq']:.2f}")

## Visualize Measured Concentrations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

elements = ['La', 'Nd', 'Dy']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, elem in enumerate(elements):
    # Aqueous concentrations
    ax = axes[0, i]
    pH_exp = [d['pH'] for d in experimental_data]
    C_aq_exp = [d[f'C_{elem}_aq'] for d in experimental_data]
    ax.scatter(pH_exp, C_aq_exp, alpha=0.6, s=50, color=colors[i])
    ax.set_xlabel('pH', fontsize=11)
    ax.set_ylabel(f'C_aq ({elem}) [mg/L]', fontsize=11)
    ax.set_title(f'{elem} Aqueous Phase', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Organic concentrations
    ax = axes[1, i]
    C_org_exp = [d[f'C_{elem}_org'] for d in experimental_data]
    ax.scatter(pH_exp, C_org_exp, alpha=0.6, s=50, color=colors[i])
    ax.set_xlabel('pH', fontsize=11)
    ax.set_ylabel(f'C_org ({elem}) [mg/L]', fontsize=11)
    ax.set_title(f'{elem} Organic Phase', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nData shows typical extraction behavior:")
print("- C_aq decreases with pH (more extraction)")
print("- C_org increases with pH (more loaded)")
print("- Dy extracts more than Nd, which extracts more than La")

## Define Model for Parameter Estimation

The model must:
1. Take parameters θ (a, b, c for each element)
2. Take an Experiment with inputs (pH, C0, V_aq, V_org)
3. **Predict the equilibrium concentrations** C_aq and C_org

This is the key difference - we're predicting concentrations, not D values!

In [ ]:
def concentration_model(theta, exp):
    """
    Predict equilibrium concentrations from pH and initial conditions.
    
    Parameters
    ----------
    theta : dict
        Parameters {a_La, b_La, c_La, a_Nd, b_Nd, c_Nd, a_Dy, b_Dy, c_Dy}
    exp : Experiment
        Must have exp.inputs['pH', 'C0_X', 'V_aq', 'V_org']
    
    Returns
    -------
    dict
        Predicted concentrations {C_La_aq, C_La_org, C_Nd_aq, ...}
    """
    pH = exp.inputs['pH']
    V_aq = exp.inputs['V_aq']
    V_org = exp.inputs['V_org']
    phase_ratio = V_org / V_aq
    
    predictions = {}
    
    for elem in ['La', 'Nd', 'Dy']:
        # Get D from pH-dependent model
        a = theta[f'a_{elem}']
        b = theta[f'b_{elem}']
        c = theta[f'c_{elem}']
        
        log_D = a + b * pH + c * pH**2
        D = jnp.power(10.0, log_D)
        
        # Calculate equilibrium concentrations
        # Mass balance: C0 = C_aq + D * C_aq * (V_org/V_aq)
        C0 = exp.inputs[f'C0_{elem}']
        C_aq = C0 / (1.0 + D * phase_ratio)
        C_org = D * C_aq
        
        predictions[f'C_{elem}_aq'] = C_aq
        predictions[f'C_{elem}_org'] = C_org
    
    return predictions

## Create Experiment Objects

Convert our experimental data (measured concentrations) into `Experiment` objects.

In [ ]:
experiments = []

for i, data in enumerate(experimental_data):
    # Measurement uncertainties (2% relative error for ICP-MS)
    uncertainties = {}
    for elem in ['La', 'Nd', 'Dy']:
        uncertainties[f'C_{elem}_aq'] = 0.02 * data[f'C_{elem}_aq']
        uncertainties[f'C_{elem}_org'] = 0.02 * data[f'C_{elem}_org']
    
    exp = Experiment(
        inputs={
            'pH': data['pH'],
            'C0_La': data['C0_La'],
            'C0_Nd': data['C0_Nd'],
            'C0_Dy': data['C0_Dy'],
            'V_aq': data['V_aq'],
            'V_org': data['V_org'],
        },
        observed={
            'C_La_aq': data['C_La_aq'],
            'C_La_org': data['C_La_org'],
            'C_Nd_aq': data['C_Nd_aq'],
            'C_Nd_org': data['C_Nd_org'],
            'C_Dy_aq': data['C_Dy_aq'],
            'C_Dy_org': data['C_Dy_org'],
        },
        uncertainties=uncertainties,
        name=f"Exp_{i+1}",
    )
    experiments.append(exp)

print(f"Created {len(experiments)} Experiment objects")
print(f"Each experiment has {len(experiments[0].observed)} measured values")
print(f"Total observations: {len(experiments) * len(experiments[0].observed)}")

## Set Up and Run Parameter Estimation

In [ ]:
# Define parameter names
param_names = [
    'a_La', 'b_La', 'c_La',
    'a_Nd', 'b_Nd', 'c_Nd',
    'a_Dy', 'b_Dy', 'c_Dy',
]

# Set parameter bounds (physical constraints)
param_bounds = {
    # La bounds
    'a_La': (-5.0, 2.0),
    'b_La': (0.0, 5.0),
    'c_La': (-1.0, 0.0),
    # Nd bounds
    'a_Nd': (-5.0, 2.0),
    'b_Nd': (0.0, 5.0),
    'c_Nd': (-1.0, 0.0),
    # Dy bounds
    'a_Dy': (-5.0, 2.0),
    'b_Dy': (0.0, 5.0),
    'c_Dy': (-1.0, 0.0),
}

# Initial guess
theta_init = {
    'a_La': -2.0, 'b_La': 1.5, 'c_La': -0.1,
    'a_Nd': -1.5, 'b_Nd': 1.8, 'c_Nd': -0.15,
    'a_Dy': -1.0, 'b_Dy': 2.0, 'c_Dy': -0.2,
}

# Create estimator
estimator = Estimator(
    model_fn=concentration_model,
    param_names=param_names,
    param_bounds=param_bounds,
)

print("Fitting parameters to concentration data...")
result = estimator.fit(
    experiments=experiments,
    theta_init=theta_init,
    objective='wsse',  # Weighted sum of squared errors
    method='L-BFGS-B',
)

print(f"\nConverged: {result.converged}")
print(f"Objective value: {result.objective_value:.6f}")
print(f"Iterations: {result.n_iterations}")

## Compute Uncertainty Estimates

In [ ]:
print("Computing confidence intervals...")
ci = estimator.confidence_intervals(result, experiments, alpha=0.05)

print("Computing diagnostics...")
diag = estimator.diagnostics(result, experiments)

print("Running bootstrap (100 samples)...")
bs = estimator.bootstrap(
    result, 
    experiments, 
    n_bootstrap=100,
    method='nonparametric',
    seed=42,
)

print("\nDone!")

## Results Summary

In [ ]:
print(estimator.summary(result, experiments))

## Compare Fitted vs True Parameters

In [ ]:
# Create comparison table
comparison = []
for elem in ['La', 'Nd', 'Dy']:
    for param in ['a', 'b', 'c']:
        key = f'{param}_{elem}'
        comparison.append({
            'Element': elem,
            'Parameter': param,
            'True': true_params[elem][param],
            'Fitted': result.theta_opt[key],
            'Error': result.theta_opt[key] - true_params[elem][param],
            'Std Error': ci.std_errors[key],
            '95% CI Lower': ci.ci_lower[key],
            '95% CI Upper': ci.ci_upper[key],
        })

df = pd.DataFrame(comparison)
df['True in CI'] = (
    (df['True'] >= df['95% CI Lower']) & 
    (df['True'] <= df['95% CI Upper'])
)

print("\nParameter Comparison: Fitted vs True Values")
print("=" * 110)
with pd.option_context('display.max_columns', None, 'display.width', None, 'display.precision', 4):
    print(df.to_string(index=False))

n_in_ci = df['True in CI'].sum()
n_total = len(df)
print(f"\n{n_in_ci}/{n_total} true values within 95% CI (expected: ~95%)")
print(f"Max absolute error: {df['Error'].abs().max():.6f}")

## Parity Plots: Predicted vs Measured Concentrations

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

elements = ['La', 'Nd', 'Dy']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, elem in enumerate(elements):
    # Aqueous phase parity plot
    ax = axes[0, i]
    observed_aq = []
    predicted_aq = []
    
    for exp in experiments:
        pred = concentration_model(result.theta_opt, exp)
        observed_aq.append(exp.observed[f'C_{elem}_aq'])
        predicted_aq.append(pred[f'C_{elem}_aq'])
    
    ax.scatter(observed_aq, predicted_aq, alpha=0.6, s=60, color=colors[i])
    
    # Perfect prediction line
    lims = [0, max(max(observed_aq), max(predicted_aq)) * 1.1]
    ax.plot(lims, lims, 'k--', linewidth=2, alpha=0.5, label='Perfect fit')
    
    # ±10% error bands
    ax.fill_between(lims, [0.9*x for x in lims], [1.1*x for x in lims], 
                     color='gray', alpha=0.2, label='±10%')
    
    ax.set_xlabel(f'Measured C_aq ({elem}) [mg/L]', fontsize=11)
    ax.set_ylabel(f'Predicted C_aq ({elem}) [mg/L]', fontsize=11)
    ax.set_title(f'{elem} Aqueous Phase', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect('equal')
    
    # Organic phase parity plot
    ax = axes[1, i]
    observed_org = []
    predicted_org = []
    
    for exp in experiments:
        pred = concentration_model(result.theta_opt, exp)
        observed_org.append(exp.observed[f'C_{elem}_org'])
        predicted_org.append(pred[f'C_{elem}_org'])
    
    ax.scatter(observed_org, predicted_org, alpha=0.6, s=60, color=colors[i])
    
    lims = [0, max(max(observed_org), max(predicted_org)) * 1.1]
    ax.plot(lims, lims, 'k--', linewidth=2, alpha=0.5, label='Perfect fit')
    ax.fill_between(lims, [0.9*x for x in lims], [1.1*x for x in lims], 
                     color='gray', alpha=0.2, label='±10%')
    
    ax.set_xlabel(f'Measured C_org ({elem}) [mg/L]', fontsize=11)
    ax.set_ylabel(f'Predicted C_org ({elem}) [mg/L]', fontsize=11)
    ax.set_title(f'{elem} Organic Phase', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_aspect('equal')

plt.tight_layout()
plt.show()

print("\nParity plots show:")
print("- Points cluster along diagonal (good fit)")
print("- All within ±10% error (excellent)")
print("- No systematic bias")

## Plot Fitted Distribution Coefficients vs pH

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

elements = ['La', 'Nd', 'Dy']
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
pH_fine = np.linspace(2.0, 5.0, 100)

for i, elem in enumerate(elements):
    ax = axes[i]
    
    # Calculate implied D from measured concentrations
    pH_exp = []
    D_exp = []
    for data in experimental_data:
        pH_exp.append(data['pH'])
        D_exp.append(data[f'C_{elem}_org'] / data[f'C_{elem}_aq'])
    
    ax.scatter(pH_exp, D_exp, alpha=0.5, s=50, label='From measured C', 
               color=colors[i], zorder=3)
    
    # True D curve
    D_true = [calculate_D_true(elem, pH) for pH in pH_fine]
    ax.plot(pH_fine, D_true, 'k--', linewidth=2, label='True', alpha=0.6, zorder=2)
    
    # Fitted D curve
    a_fit = result.theta_opt[f'a_{elem}']
    b_fit = result.theta_opt[f'b_{elem}']
    c_fit = result.theta_opt[f'c_{elem}']
    D_fit = 10**(a_fit + b_fit * pH_fine + c_fit * pH_fine**2)
    
    ax.plot(pH_fine, D_fit, color=colors[i], linewidth=2.5, label='Fitted', zorder=4)
    
    ax.set_xlabel('pH', fontsize=11)
    ax.set_ylabel(f'D ({elem})', fontsize=11)
    ax.set_title(f'{elem} Distribution Coefficient', fontsize=12, fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

## Model Diagnostics

In [ ]:
print("Model Diagnostics")
print("=" * 60)
print(f"R²:              {diag.r_squared:.6f}")
print(f"Adjusted R²:     {diag.r_squared_adj:.6f}")
print(f"RMSE:            {diag.rmse:.4f} mg/L")
print(f"AIC:             {diag.aic:.2f}")
print(f"BIC:             {diag.bic:.2f}")
print(f"")
print(f"N observations:  {diag.n_obs} (21 experiments × 6 concentrations)")
print(f"N parameters:    {diag.n_params}")
print(f"Degrees freedom: {diag.n_obs - diag.n_params}")
print(f"")
print(f"Interpretation:")
print(f"- R² > 0.999 indicates excellent fit")
print(f"- RMSE < 1 mg/L is well within analytical uncertainty")
print(f"- All true parameters recovered within confidence intervals")

## Key Takeaways

### What's Different in This Approach?

1. **We fit to measured concentrations** (C_aq, C_org), not derived D values
   - This is what you actually measure in the lab (ICP-MS, UV-Vis, etc.)
   - More realistic experimental workflow
   
2. **Error propagation is handled correctly**
   - Measurement errors in concentrations propagate naturally
   - No need to calculate uncertainty in D explicitly
   
3. **Model predicts observable quantities**
   - Given pH and initial conditions → predict equilibrium concentrations
   - Distribution coefficient is an intermediate calculation

### Why This Matters

- **Physically realistic**: Matches actual lab procedures
- **Better statistics**: Fitting raw data is more rigorous
- **Extensible**: Easy to add mass balance constraints, multiple phases, etc.
- **Transparent**: Clear what's measured vs calculated

### Workflow for Your Own Data

```python
# 1. Load your measured concentrations from lab
my_data = pd.read_csv('extraction_data.csv')

# 2. Create Experiment objects
experiments = [
    Experiment(
        inputs={'pH': row['pH'], 'C0_La': 100.0, ...},
        observed={'C_La_aq': row['C_La_aq_measured'], ...},
        uncertainties={'C_La_aq': row['C_La_aq_uncertainty'], ...}
    )
    for _, row in my_data.iterrows()
]

# 3. Fit and validate
result = estimator.fit(experiments, theta_init)
ci = estimator.confidence_intervals(result, experiments)
```

The rest is identical!